<a href="https://colab.research.google.com/github/nandakumar3261/ai-mentor-portfolio-Nandakumar/blob/main/Day6_PlacementProcessor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-genai pydantic

In [ ]:
import os

if 'GEMINI_API_KEY' in os.environ:
    del os.environ['GEMINI_API_KEY']

print("API key removed")

API key removed


In [ ]:
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [ ]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [ ]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    """Extract a Resume JSON from raw text. Retries once on schema fail."""
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == max_retries:
                raise
            fix_prompt = f'Fix this JSON to match schema. Errors: {e}. Original: {resp.text}'
            resp = client.models.generate_content(
                model='gemini-2.5-flash', contents=fix_prompt,
                config={'response_mime_type': 'application/json',
                        'response_schema': Resume.model_json_schema()})
            return Resume.model_validate_json(resp.text)

In [ ]:
with open('/content/sample_data/sample_resumes.txt') as f:
    resumes = [r.strip() for r in f.read().split('---') if r.strip()]
print(f'Loaded {len(resumes)} sample résumés')

results = []
errors = []
for i, r in enumerate(resumes):
    try:
        parsed = extract_resume(r)
        results.append(parsed)
        print(f'  [{i+1}] {parsed.name} — {len(parsed.skills)} skills')
    except Exception as e:
        errors.append((i, e))
        print(f'  [{i+1}] FAILED: {type(e).__name__}: {str(e)[:120]}')

print(f'\n{len(results)}/5 succeeded, {len(errors)} failed')

Loaded 5 sample résumés
  [1] Ravi Kumar — 6 skills
  [2] Sneha Reddy — 6 skills
  [3] Arun Pillai — 8 skills
  [4] Priya Nair — 5 skills
  [5] Karthik Sharma — 5 skills

5/5 succeeded, 0 failed


In [ ]:
try:
    bad = extract_resume('')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print(f'Empty input: {type(e).__name__}: {str(e)[:200]}')

# Whitespace only
try:
    bad = extract_resume('   \n\n   ')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print(f'Whitespace input: {type(e).__name__}: {str(e)[:200]}')

# Garbage non-résumé text
try:
    bad = extract_resume('the quick brown fox jumps over the lazy dog')
    print('Garbage input:', bad.model_dump_json())
except Exception as e:
    print(f'Garbage input: {type(e).__name__}: {str(e)[:200]}')

Unexpected success: {"name":"John Doe","email":"john.doe@example.com","phone":null,"education":[{"degree":"B.S. Computer Science","institution":"University of Placeholder","year":2020}],"skills":["Python","JavaScript","Data Analysis"],"projects":["Automated Reporting System","E-commerce Website Redesign"],"experience_years":5.0}
Unexpected success: {"name":"","email":"","phone":null,"education":[],"skills":[],"projects":[],"experience_years":0.0}
Garbage input: {"name":"Unknown","email":"unknown@example.com","phone":null,"education":[],"skills":[],"projects":[],"experience_years":0.0}


In [ ]:
class JD(BaseModel):
    company: str
    role: str
    must_have_skills: List[str]
    nice_to_have_skills: List[str] = []
    min_cgpa: Optional[float] = None
    locations: List[str] = []
    package_lpa: Optional[float] = None

In [ ]:
import requests
from bs4 import BeautifulSoup
import pathlib, json

def fetch_jd(url, max_chars=6000):
    """Fetch JD URL and return clean text. Returns None on block / failure."""
    try:
        r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, 'html.parser')
        # Remove script and style tags
        for tag in soup(['script', 'style']):
            tag.decompose()
        return soup.get_text(separator='\n', strip=True)[:max_chars]
    except Exception as e:
        print(f'  Scrape failed for {url}: {e}')
        return None

# Test on one URL
test_url = 'https://amazon.jobs/en/jobs/10421853/data-engineer-shiptech-analytics'
text = fetch_jd(test_url)
if text:
    print(f'Got {len(text)} chars')
    print(text[:300])
else:
    print('Scrape blocked. Will use cached set.')

Got 5093 chars
Data Engineer, ShipTech Analytics - Job ID: 10421853 | Amazon.jobs
Skip to main content
×
Home
Teams
Locations
Job categories
My career
My applications
My profile
Account security
Settings
Sign out
Resources
Accommodations
Benefits
Inclusive experiences
How We Hire
Leadership principles
Working at A


In [ ]:
def normalise_jd(text: str) -> JD:
    """Send JD text to Gemini, get structured JD JSON back."""
    resp = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=f'Extract a JD JSON from this text:\n\n{text}',
        config={
            'response_mime_type': 'application/json',
            'response_schema': JD.model_json_schema(),
        },
    )
    return JD.model_validate_json(resp.text)

# Test on one JD text
if text:
    jd = normalise_jd(text)
    print(jd.model_dump_json(indent=2))

{
  "company": "Amazon",
  "role": "Data Engineer, ShipTech Analytics",
  "must_have_skills": [
    "SQL",
    "Data modeling",
    "Data warehousing",
    "ETL pipelines",
    "Python",
    "KornShell",
    "Scala"
  ],
  "nice_to_have_skills": [
    "Hadoop",
    "Hive",
    "Spark",
    "EMR",
    "AWS Redshift",
    "AWS S3",
    "AWS Glue",
    "AWS Kinesis",
    "AWS FireHose",
    "AWS Lambda",
    "AWS IAM"
  ],
  "min_cgpa": null,
  "locations": [
    "Hyderabad"
  ],
  "package_lpa": null
}


In [ ]:
import json, pathlib

URLS = [
    # Paste your 5 assigned URLs here
    'https://amazon.jobs/en/jobs/10399137/maintenance-technician',
    'https://amazon.jobs/en/jobs/3206819/mechatronics-robotics-tech',
    'https://amazon.jobs/en/jobs/10393375/associate-systems-engineer-windows-region-services',
    'https://amazon.jobs/en/jobs/10429156/applied-scientist-rbs-tech',
    'https://amazon.jobs/en/jobs/10429151/salesforce-developer-dsp-tech-last-mile',
]

CACHE = pathlib.Path('/content/sample_data/jds_cached.jsonl')
USE_CACHE = False   # set True if scraping is blocked

jds = []

if USE_CACHE and CACHE.exists():
    print(f'Using cached JDs from {CACHE}')
    for line in CACHE.read_text().splitlines():
        jds.append(JD.model_validate_json(line))
else:
    for url in URLS:
        text = fetch_jd(url)
        if text is None:
            continue
        try:
            jd = normalise_jd(text)
            jds.append(jd)
            print(f'  ✓ {jd.company} — {jd.role}')
        except Exception as e:
            print(f'  ✗ {url}: {e}')

print(f'\nProcessed {len(jds)} JDs')

# Inspect first 3
for jd in jds[:3]:
    print(f'\n{jd.company} - {jd.role}')
    print(f'  Must: {jd.must_have_skills}')
    print(f'  Nice: {jd.nice_to_have_skills}')
    print(f'  CGPA: {jd.min_cgpa}, LPA: {jd.package_lpa}')

  ✓ Amazon — Maintenance Technician
  ✓ Amazon — Mechatronics & Robotics Tech
  ✓ Amazon — Associate Systems Engineer - Windows , Region Services
  ✓ Amazon — Applied Scientist
  ✓ Amazon — Salesforce Developer

Processed 5 JDs

Amazon - Maintenance Technician
  Must: ['Formal Trade Qualifications (Electrotechnology, Engineering, or Mechanical Trade)', 'Experience of planned preventative maintenance systems', 'Experience fault finding within Material Handling Equipment/Automation systems', 'Ability to read and understand electrical/mechanical drawings', 'Experience of conveyor maintenance, motor controllers/inverters', 'Experience of working to appropriate health & safety standards and regulations']
  Nice: ['Experience with CMMS systems', 'Experience with sortation machines', 'Experience with maintaining/configuring bar code scanners', 'Experience with print and apply machines']
  CGPA: None, LPA: None

Amazon - Mechatronics & Robotics Tech
  Must: ['Microsoft Office products and appl

In [ ]:
OUT = pathlib.Path('/content/sample_data/jds.jsonl.rtf')
OUT.parent.mkdir(exist_ok=True)
with open(OUT, 'w') as f:
    for jd in jds:
        f.write(jd.model_dump_json() + '\n')
print(f'Wrote {len(jds)} JDs to {OUT}')

# Verify the file
with open(OUT) as f:
    for line in f:
        d = json.loads(line)
        print(f'  {d["company"]:20} | {d["role"]:30} | {len(d["must_have_skills"])} must-haves')

Wrote 5 JDs to /content/sample_data/jds.jsonl.rtf
  Amazon               | Maintenance Technician         | 6 must-haves
  Amazon               | Mechatronics & Robotics Tech   | 6 must-haves
  Amazon               | Associate Systems Engineer - Windows , Region Services | 9 must-haves
  Amazon               | Applied Scientist              | 7 must-haves
  Amazon               | Salesforce Developer           | 12 must-haves
